# Part 1. Equation of a Slime

How many late days are you using for this assignment?

In [1]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import numpy as np



## 1. Loading the dataset

In [2]:
# Using pandas load the dataset
dataS = pd.read_csv("science_data_large.csv")
# Output the first 15 rows of the data
dataS.head(15)
# Display a summary of the table information (number of datapoints, etc.)
dataS.info()

#dataS["Size nm^3"].head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


## 2. Splitting the dataset

In [3]:
# Take the pandas dataset and split it into our features (X) and label (y)
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 

X = dataS.drop('Size nm^3', axis=1) 
y = dataS['Size nm^3']

# Now we can split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Now X_train, y_train can be used for training a model
# and X_test, y_test can be used for testing the model


## 3. Perform a Linear Regression

In [4]:
# Use sklearn to train a model on the training set
reg = LinearRegression().fit(X_train, y_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample_point = pd.DataFrame([[500, 500]], columns=['Temperature °C','Mols KCL'])

predicted_output = reg.predict(sample_point)
print("Predicted output for sample point [500, 500]:", predicted_output[0])

# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means
score = reg.score(X_train, y_train)
print("Model score (R^2):", score)
# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = reg.coef_
intercept = reg.intercept_

print("Coefficients:", coefficients)
print("Intercept:", intercept)

Predicted output for sample point [500, 500]: 540029.260345451
Model score (R^2): 0.8610082946223788
Coefficients: [ 866.14641337 1032.69506649]
Intercept: -409391.47958340764


Write the linear equation of a slime: (example equation: $E = mc^2$)


\[
h(x) = 866.146 \cdot temp + 1032.695 \cdot mols - 409391.480
\]

Report on score and explain meaning:


An R2=1 means that the data is perfectly correlated, meaning standard error is 0. When performing linear regression, you want the value of R2 to be as close to 1 as possible. In our case we have a score of 0.86 meaning we have some error, it is not perfectly correlated, but we can estimate pretty well what our slime might grow given our input values. 

## 4. Use Cross Validation

In [5]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42
X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)

# Initialize the model
model = LinearRegression()

# Use cross_val_score to evaluate the model with 5 splits
scores = cross_val_score(model, X, y, cv=5)  # n_splits is determined by cv parameter

print("Cross-validation scores for each fold:", scores)
print("Mean cross-validation score (R^2):", np.mean(scores))

# Report on their finding and their significance

Cross-validation scores for each fold: [0.99999945 0.99999847 0.99999862 0.99999911 0.9999988 ]
Mean cross-validation score (R^2): 0.9999988909271998


Write findings here:

The score is so nearly close to 1 across the board and the mean very much so rounds to 1. This means that despite the subset the linear regression is consistant in its guesses. This probably means linear regression is not the best fit because in all possible permutations it is always close to other permutations but not all that close to actually guessing the correct value for y. This is like precision vs accuracy. It is very precise but not super accurate. That's the best way I can think to explain that.

## 5. Using Polynomial Regression

In [6]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
model = LinearRegression()


X_train_poly = poly.fit_transform(X_train)  # Transform training set
X_test_poly = poly.transform(X_test)  # Transform testing set


# Fit the model on the training data
model.fit(X_train_poly, y_train)

# Calculate R^2 score on the training data
train_r2_score = model.score(X_train_poly, y_train)
print("R^2 score for the training set:", train_r2_score)



scores = cross_val_score(model, X_poly, y, cv=5)
print("Cross-validation scores for each fold:", scores)
print("Mean cross-validation score (R^2):", np.mean(scores))
model.fit(X_poly, y)

coefficients = model.coef_
intercept = model.intercept_

terms = ""
for i in range(len(coefficients)):
    if coefficients[i] != 0:  
        if i == 0:
            terms += f"{coefficients[i]:.4f}"
        else:
            if i == 1:
                terms += f" + {coefficients[i]:.4f} * temp"
            elif i == 2:
                terms += f" + {coefficients[i]:.4f} * mols"
            elif i == 3:
                terms += f" + {coefficients[i]:.4f} * temp^2"
            elif i == 4:
                terms += f" + {coefficients[i]:.4f} * temp * mols"
            elif i == 5:
                terms += f" + {coefficients[i]:.4f} * mols^2"

equation = f"h(x) = {terms}"
print("Polynomial regression equation:", equation)
# Report on the metrics and output the resultant equation as you did in Part 3.

R^2 score for the training set: 1.0
Cross-validation scores for each fold: [0.99999945 0.99999852 0.99999833 0.99999915 0.99999869]
Mean cross-validation score (R^2): 0.9999988283872593
Polynomial regression equation: h(x) =  + 87.7201 * temp + 74.0752 * mols + 0.0081 * temp^2 + 0.0235 * temp * mols + 0.0053 * mols^2


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

h(x) =  + 87.7201 * temp + 74.0752 * mols + 0.0081 * temp^2 + 0.0235 * temp * mols + 0.0053 * mols^2

Report on the score and interpret:

This is perfectly fitting the data. Because the score is 1, it means it is accurately able account for every point. 

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [7]:
# Load the dataset. Then train and evaluate the classification models.
dataK = pd.read_csv("ckd_feature_subset.csv")
X = dataK.drop('Target_ckd', axis=1) 
y = dataK['Target_ckd']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Create a dictionary to store the results
results = {}

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg_scores = cross_val_score(log_reg, X, y, cv=kf)
results['Logistic Regression'] = (np.mean(log_reg_scores), np.std(log_reg_scores))

# Support Vector Machines
svc = SVC()
svc_scores = cross_val_score(svc, X, y, cv=kf)
results['Support Vector Machine'] = (np.mean(svc_scores), np.std(svc_scores))

# k-Nearest Neighbors
knn = KNeighborsClassifier()
knn_scores = cross_val_score(knn, X, y, cv=kf)
results['k-Nearest Neighbors'] = (np.mean(knn_scores), np.std(knn_scores))

#  Default Neural Networks Configuration
nn1 = MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000, random_state=42)
nn1_scores = cross_val_score(nn1, X, y, cv=kf)
results['Neural Network 1'] = (np.mean(nn1_scores), np.std(nn1_scores))


#results_df = pd.DataFrame(results, index=['Average Accuracy', 'Standard Deviation']).T
#print(results_df)





/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/pytho

## Results and Conclusion for Classification Experiments

In [8]:
# Experiments with Neural Network.

#2: Increase layers and neurons
nn2 = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42)
nn2_scores = cross_val_score(nn2, X, y, cv=kf)
results['Neural Network 2'] = (np.mean(nn2_scores), np.std(nn2_scores))


#3: Changed activation function
nn3 = MLPClassifier(hidden_layer_sizes=(10,), activation='tanh', max_iter=1000, random_state=42)
nn3_scores = cross_val_score(nn3, X, y, cv=kf)
results['Neural Network 3'] = (np.mean(nn3_scores), np.std(nn3_scores))

nn4 = MLPClassifier(hidden_layer_sizes=(10,), activation='logistic', max_iter=1000, random_state=42)
nn4_scores = cross_val_score(nn3, X, y, cv=kf)
results['Neural Network 4'] = (np.mean(nn4_scores), np.std(nn4_scores))

results_df = pd.DataFrame(results, index=['Average Accuracy', 'Standard Deviation']).T
print(results_df)


/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/codespace/.local/lib/pytho

                        Average Accuracy  Standard Deviation
Logistic Regression             0.856559            0.066269
Support Vector Machine          0.928172            0.047601
k-Nearest Neighbors             0.927957            0.052440
Neural Network 1                0.908602            0.052041
Neural Network 2                0.954624            0.043696
Neural Network 3                0.928387            0.055305
Neural Network 4                0.928387            0.055305


/home/codespace/.local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


## Results and Conclusion for Neural Network Experiments

Only the logistic regression was lower than 0.9 average accuracy, making it the least trustworthy model in predicting data. The logistic regression perhaps I not use in this case because I would look for an accuracy that exceeds 0.9. The second neural network which was tested using the hidden layer size (50, 50) performed the best of all of the models by far, exceeding 0.95 which is a fairly accurate model. The rest performed almost equally with around 0.928 accuracy. I would definitely choose the second neural network in this case. 